# Analyse du trafic IP — ELE8704

## 1. Préparation des données

In [ ]:
from pathlib import Path
from collections import Counter
import json
import re
import shutil
import subprocess
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.metrics import (
    classification_report, ConfusionMatrixDisplay,
    accuracy_score, f1_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (9, 4.5)
SEED = 1234
rng = np.random.default_rng(SEED)

RACINE = Path.cwd()                # Ouvrir le notebook depuis son dossier racine
DOSSIER_PCAP = RACINE / "data" / "raw"
DOSSIER_CSV = RACINE / "data" / "processed"
DOSSIER_TABLEAUX = RACINE / "outputs" / "tables"
DOSSIER_FIGURES = RACINE / "outputs" / "figures"
for dossier in (DOSSIER_PCAP, DOSSIER_CSV, DOSSIER_TABLEAUX, DOSSIER_FIGURES):
    dossier.mkdir(parents=True, exist_ok=True)

# Paramètres des analyses
NB_BOOTSTRAP = 1000
MAX_POINTS_AJUSTEMENT = 5000
MAX_PAQUETS_ML = None   

# le numéro de "tcp.stream" ou "udp.stream" se trouve dans Wireshark.
FLUX_CONFIRMES = {
    
     "googlemeet_ethernet_r01": [("UDP", 0)],
     "googlemeet_ethernet_r02": [("UDP", 0)],
     "googlemeet_ethernet_r03": [("UDP", 0)],
     "googlemeet_wifi_r01": [("UDP", 0)],
     "googlemeet_wifi_r02": [("UDP", 0)],
     "googlemeet_wifi_r03": [("UDP", 0)],
     "lapack_ethernet_r01": [("TCP", 5)],
     "lapack_wifi_r01": [("TCP", 10)],
     "lapack_wifi_r02": [("TCP", 9)],
     "lapack_wifi_r03": [("TCP", 5)],
     "youtube_ethernet_r01": [("UDP", 7)], 
     "youtube_ethernet_r02": [("UDP", 4),("UDP", 19) ], 
     "youtube_ethernet_r03": [("UDP", 0)], 
     "youtube_wifi_r01": [("UDP", 22)], 
     "youtube_wifi_r02": [("UDP", 4)], 
     "youtube_wifi_r03": [("UDP", 16)], 

          
          
     

}

print("Racine de travail :", RACINE)

In [ ]:
# Types de paquets, sélection des flux et interarrivées par flux dirigé
CONTROLES_IDENTIFIABLES = {"dns", "mdns", "bootp", "dhcpv6", "ntp", "stun", "icmp", "icmpv6", "igmp"}


def etiqueter_paquets(df):
    df = df.copy()
    piles = df["pile_protocoles"].fillna("").astype(str).str.lower()
    est_controle = piles.map(lambda p: bool(set(p.split(":")) & CONTROLES_IDENTIFIABLES))
    tcp = df["transport"].eq("TCP")
    udp = df["transport"].eq("UDP")
    df["type_paquet"] = "indetermine"
    df.loc[tcp & df["tcp_charge"].eq(0), "type_paquet"] = "tcp_controle"
    df.loc[tcp & df["tcp_charge"].gt(0), "type_paquet"] = "tcp_donnees"
    df.loc[udp & df["udp_longueur"].gt(8), "type_paquet"] = "udp_charge_indeterminee"
    df.loc[est_controle, "type_paquet"] = "controle_identifiable"

    # Le champ UDP « charge utile » peut aussi contenir du contrôle QUIC chiffré.
    df["charge_utile_octets"] = np.where(
        tcp, df["tcp_charge"],
        np.where(udp, (df["udp_longueur"] - 8).clip(lower=0), np.nan)
    )
    df["flux_dirige"] = (df["transport"] + "|" + df["ip_src"] + "|" +
                         df["port_src"].fillna(-1).astype(int).astype(str) + "|" +
                         df["ip_dst"] + "|" +
                         df["port_dst"].fillna(-1).astype(int).astype(str))
    df["interarrivee_s"] = df.groupby("flux_dirige", sort=False)["temps_s"].diff()
    df.loc[df["interarrivee_s"] < 0, "interarrivee_s"] = np.nan
    secondes = np.floor(df["temps_s"] - df["temps_s"].min()).astype(int)
    df["paquets_par_s"] = secondes.map(secondes.value_counts())
    return df


def selectionner_flux(df, capture_id):
    choix = FLUX_CONFIRMES.get(capture_id, [])
    if not choix:
        return df.iloc[0:0].copy()  # Ne pas inventer de « flux utile »
    masque = np.zeros(len(df), dtype=bool)
    for transport, numero in choix:
        masque |= (df["transport"].eq(transport.upper()) &
                   df["stream_id"].eq(int(numero))).to_numpy()
    return df.loc[masque].copy()


def paquets_information(df):
    return df[df["type_paquet"].isin(["tcp_donnees", "udp_charge_indeterminee"])].copy()

## 2. Lecture des fichiers PCAP

In [ ]:
CHAMPS_TSHARK = {
    "frame.number":"numero", "frame.time_epoch":"temps_s", "frame.len":"taille_trame",
    "frame.cap_len":"taille_capture", "frame.protocols":"pile_protocoles",
    "ip.src":"ipv4_src", "ip.dst":"ipv4_dst", "ip.len":"ipv4_taille",
    "ipv6.src":"ipv6_src", "ipv6.dst":"ipv6_dst", "ipv6.plen":"ipv6_payload_len",
    "tcp.srcport":"tcp_port_src", "tcp.dstport":"tcp_port_dst", "tcp.len":"tcp_charge",
    "tcp.flags.syn":"tcp_syn", "tcp.flags.fin":"tcp_fin", "tcp.flags.rst":"tcp_rst",
    "tcp.flags.ack":"tcp_ack", "tcp.stream":"tcp_stream",
    "udp.srcport":"udp_port_src", "udp.dstport":"udp_port_dst",
    "udp.length":"udp_longueur", "udp.stream":"udp_stream",
    "rtp.ssrc":"rtp_ssrc", "rtp.seq":"rtp_seq", "rtp.timestamp":"rtp_timestamp"
}


def localiser_tshark():
    chemin = shutil.which("tshark")
    if chemin:
        return chemin
    candidats = [Path(r"C:\Program Files\Wireshark\tshark.exe"),
                 Path(r"C:\Program Files (x86)\Wireshark\tshark.exe")]
    for candidat in candidats:
        if candidat.exists():
            return str(candidat)
    return None


def obtenir_champs_tshark(executable):
    resultat = subprocess.run([executable, "-G", "fields"], capture_output=True,
                             text=True, encoding="utf-8", errors="replace", check=True)
    # Format ligne : F <tab> nom <tab> abréviation <tab> type ...
    return {col[2] for ligne in resultat.stdout.splitlines()
            if ligne.startswith("F\t") and len(col := ligne.split("\t")) > 2}


def extraire_pcap(fichier, executable, disponibles):
    champs = [(nom, alias) for nom, alias in CHAMPS_TSHARK.items() if nom in disponibles]
    obligatoires = {"frame.time_epoch", "frame.number", "frame.len"}
    if not obligatoires.issubset({nom for nom, _ in champs}):
        raise RuntimeError("Champs fondamentaux absents de TShark.")
    commande = [executable, "-r", str(fichier), "-Y", "ip or ipv6", "-T", "fields",
                "-E", "header=y", "-E", "separator=,", "-E", "quote=d",
                "-E", "occurrence=f"]
    for nom, _ in champs:
        commande += ["-e", nom]
    sortie = DOSSIER_CSV / f"{fichier.stem}_extrait.csv"
    with sortie.open("w", encoding="utf-8", newline="") as dest:
        proc = subprocess.run(commande, stdout=dest, stderr=subprocess.PIPE,
                              text=True, encoding="utf-8", errors="replace")
    if proc.returncode != 0:
        sortie.unlink(missing_ok=True)
        raise RuntimeError(f"TShark a échoué pour {fichier.name}: {proc.stderr[-900:]}")
    df = pd.read_csv(sortie, dtype=str, low_memory=False)
    df.columns = [alias for _, alias in champs]
    for alias in CHAMPS_TSHARK.values():
        if alias not in df.columns:
            df[alias] = np.nan
    return df


def convertir_numerique(df):
    non_numeriques = {"pile_protocoles", "ipv4_src", "ipv4_dst", "ipv6_src",
                     "ipv6_dst", "rtp_ssrc"}
    for col in set(CHAMPS_TSHARK.values()) - non_numeriques:
        if col in df:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    for col in ("ipv4_src", "ipv4_dst", "ipv6_src", "ipv6_dst"):
        df[col] = df[col].fillna("").astype(str)
    df["ip_src"] = df["ipv4_src"].where(df["ipv4_src"] != "", df["ipv6_src"])
    df["ip_dst"] = df["ipv4_dst"].where(df["ipv4_dst"] != "", df["ipv6_dst"])
    df["taille_ip"] = df["ipv4_taille"].where(df["ipv4_taille"].notna(),
                                             40.0 + df["ipv6_payload_len"])
    df["transport"] = np.where(df["tcp_port_src"].notna(), "TCP",
                     np.where(df["udp_port_src"].notna(), "UDP", "AUTRE"))
    df["port_src"] = df["tcp_port_src"].where(df["transport"] == "TCP", df["udp_port_src"])
    df["port_dst"] = df["tcp_port_dst"].where(df["transport"] == "TCP", df["udp_port_dst"])
    df["stream_id"] = df["tcp_stream"].where(df["transport"] == "TCP", df["udp_stream"])
    df["taille_ip"] = df["taille_ip"].where(df["taille_ip"] > 0)
    df = df.dropna(subset=["temps_s", "taille_ip", "ip_src", "ip_dst"]).copy()
    df = df.sort_values(["temps_s", "numero"], kind="mergesort").reset_index(drop=True)
    df["trame_tronquee"] = df["taille_capture"].lt(df["taille_trame"]).fillna(False)
    return df


TSHARK = localiser_tshark()

In [ ]:
MODELE_NOM = re.compile(r"^(youtube|googlemeet|lapack)_(wifi|ethernet)_r(\d+)$", re.I)

def identifier_capture(path):
    correspondance = MODELE_NOM.match(path.stem)
    if not correspondance:
        return None
    appli, reseau, repetition = correspondance.groups()
    return {"capture_id": path.stem, "application": appli.lower(),
            "reseau": reseau.lower(), "repetition": int(repetition), "fichier": path}

fichiers = sorted([p for p in DOSSIER_PCAP.iterdir()
                   if p.suffix.lower() in {".pcap", ".pcapng"}])
inventaire = [objet for chemin in fichiers if (objet := identifier_capture(chemin))]
for chemin in fichiers:
    if identifier_capture(chemin) is None:
        print("Fichier ignoré (nom non conforme) :", chemin.name)

if not inventaire:
    print("Aucune capture trouvée. Placer les fichiers *.pcap dans", DOSSIER_PCAP)
else:
    display(pd.DataFrame(inventaire).assign(fichier=lambda x: x.fichier.map(lambda p: p.name)))
    print("Nombre de captures valides :", len(inventaire))

In [ ]:
captures_brutes = {}
if not inventaire:
    print("Aucune capture PCAP disponible.")
elif TSHARK is None:
    print("TShark introuvable : installer Wireshark/TShark pour lire les PCAP.")
else:
    disponibles = obtenir_champs_tshark(TSHARK)
    manquants = sorted(set(CHAMPS_TSHARK) - disponibles)
    if manquants:
        print("Champs TShark absents, remplacés par NaN :", manquants)
    for entree in inventaire:
        try:
            df = convertir_numerique(extraire_pcap(entree["fichier"], TSHARK, disponibles))
            captures_brutes[entree["capture_id"]] = df
            duree = df.temps_s.max() - df.temps_s.min() if len(df) else 0
            print(f"{entree['capture_id']}: {len(df):,} IP, durée observée {duree:.1f} s")

        except (RuntimeError, ValueError, pd.errors.EmptyDataError) as exc:
            print("Erreur d'extraction :", exc)

In [ ]:
# Statistiques : flux confirmés ; ML : tous les paquets IPv4/IPv6 de chaque capture
captures = {}
captures_tous_ip = {}
for info in inventaire:
    identifiant = info["capture_id"]
    if identifiant not in captures_brutes:
        continue
    tous = etiqueter_paquets(captures_brutes[identifiant])
    for cle in ("capture_id", "application", "reseau", "repetition"):
        tous[cle] = info[cle]
    captures_tous_ip[identifiant] = tous
    utile = selectionner_flux(tous, identifiant)
    if utile.empty:
        print(f"{identifiant} : aucun paquet du flux confirmé (analyse statistique ignorée)")
    else:
        captures[identifiant] = utile
        print(f"{identifiant} : {len(utile):,} paquets sélectionnés / {len(tous):,} IP capturés")

## 4. Taille des paquets de données — moments, histogrammes et ajustements

In [ ]:
LOIS = {"Exponentielle": stats.expon, "Normale": stats.norm, "Uniforme": stats.uniform}


def moments(valeurs):
    x = np.asarray(valeurs, dtype=float)
    x = x[np.isfinite(x)]
    if not len(x):
        return {"n": 0, "m1": np.nan, "m2": np.nan,
                "variance": np.nan, "ecart_type": np.nan}
    m1 = float(np.mean(x))
    return {"n": len(x), "m1": m1, "m2": float(np.mean(x**2)),
            "variance": float(np.mean((x - m1)**2)),
            "ecart_type": float(np.std(x, ddof=0))}


def ajuster_loi(x, loi):
    if loi == "Exponentielle":
        return LOIS[loi].fit(x, floc=0)
    return LOIS[loi].fit(x)


def statistique_ks(x, loi, parametres):
    # KS calculée directement pour limiter le coût de la simulation.
    cdf = LOIS[loi].cdf(np.sort(x), *parametres)
    k = np.arange(1, len(x) + 1)
    return float(max(np.max(k / len(x) - cdf), np.max(cdf - (k - 1) / len(x))))


def ajustements_bootstrap(valeurs, graine):
    x = np.asarray(valeurs, dtype=float)
    x = x[np.isfinite(x) & (x > 0)]  # Ajustement continu : valeurs positives.
    generateur = np.random.default_rng(graine)
    if len(x) > MAX_POINTS_AJUSTEMENT:
        x = generateur.choice(x, MAX_POINTS_AJUSTEMENT, replace=False)
    resultats = {}
    for nom in LOIS:
        if len(x) < 30 or np.ptp(x) <= 1e-12:
            resultats[nom] = {"D": np.nan, "p": np.nan, "params": None}
            continue
        try:
            parametres = ajuster_loi(x, nom)
            D = statistique_ks(x, nom, parametres)
            depassements = 0
            for _ in range(NB_BOOTSTRAP):
                simule = LOIS[nom].rvs(*parametres, size=len(x), random_state=generateur)
                par_simule = ajuster_loi(simule, nom)
                depassements += statistique_ks(simule, nom, par_simule) >= D
            resultats[nom] = {"D": D, "p": (1 + depassements) / (1 + NB_BOOTSTRAP),
                              "params": parametres}
        except (ValueError, FloatingPointError, RuntimeError) as exc:
            warnings.warn(f"{nom} : {exc}")
            resultats[nom] = {"D": np.nan, "p": np.nan, "params": None}
    return resultats


def analyser_serie(valeurs, categorie, identifiant, variable, graine):
    x = np.asarray(valeurs, dtype=float)
    x = x[np.isfinite(x) & (x >= 0)]
    ligne = {"niveau": categorie, "identifiant": identifiant, "variable": variable,
             **moments(x), "fraction_zero": float(np.mean(x == 0)) if len(x) else np.nan,
             "n_ajustement": int(np.sum(x > 0))}
    lois = ajustements_bootstrap(x, graine)
    for nom, resultat in lois.items():
        suffixe = nom.lower()
        ligne[f"D_{suffixe}"] = resultat["D"]
        ligne[f"p_{suffixe}"] = resultat["p"]
    p_admissibles = {nom: val["D"] for nom, val in lois.items() if np.isfinite(val["D"])}
    ligne["meilleur_KS"] = min(p_admissibles, key=p_admissibles.get) if p_admissibles else None
    return ligne, lois


def tracer_ajustement(ax, valeurs, lois, titre, unite):
    x = np.asarray(valeurs, dtype=float)
    x = x[np.isfinite(x) & (x > 0)]
    if not len(x):
        ax.set_title(titre + " — données absentes")
        return
    plafond = float(np.quantile(x, 0.995))
    if plafond <= 0:
        plafond = float(np.max(x))
    x_affiche = x[x <= plafond]
    ax.hist(x_affiche, bins=min(55, max(10, int(np.sqrt(len(x_affiche))))),
            density=True, alpha=0.36, label="Observé")
    xx = np.linspace(0, max(plafond, 1e-10), 350)
    for nom, resultat in lois.items():
        if resultat["params"] is not None:
            ax.plot(xx, LOIS[nom].pdf(xx, *resultat["params"]),
                    label=f"{nom} (p={resultat['p']:.3f})", linewidth=1.6)
    ax.set(xlabel=unite, ylabel="Densité", title=titre, xlim=(0, plafond * 1.02))
    ax.legend(fontsize=8)


def enregistrer_figure(fig, nom, afficher=False):
    fig.tight_layout()
    fig.savefig(DOSSIER_FIGURES / nom, dpi=150)
    if afficher:
        plt.show()
    else:
        plt.close(fig)


def collecter_series(tables, variable, conversion=1):

    morceaux = []

    for table in tables:

        # Utiliser tous les paquets sans distinction de type
        info = table.copy()

        # Trier les paquets chronologiquement
        info = info.sort_values("temps_s")

        if variable == "interarrivee_s":

            # Calculer les temps d'interarrivée entre tous
            # les paquets appartenant au même flux dirigé
            valeurs = (
                info.groupby("flux_dirige", sort=False)["temps_s"]
                .diff()
                .to_numpy(float)
            )

        else:

            # Extraire la variable pour tous les paquets
            valeurs = pd.to_numeric(
                info[variable],
                errors="coerce"
            ).to_numpy(float)

        morceaux.append(valeurs * conversion)

    return np.concatenate(morceaux) if morceaux else np.array([])

In [ ]:
# ------------------------------------------------------------
# Analyse de la taille des datagrammes IP
# Tous les paquets sont considérés sans distinction de type
# ------------------------------------------------------------

resultats_taille = []
par_configuration = {}

# Regrouper les captures par application et type de réseau
for identifiant, table in captures.items():

    par_configuration.setdefault(
        (table["application"].iat[0], table["reseau"].iat[0]),
        []
    ).append(table)


# ------------------------------------------------------------
# 1. Analyse de chaque répétition
# ------------------------------------------------------------

for identifiant, table in captures.items():

    fig, ax = plt.subplots(figsize=(9, 4.5))

    variable = "taille_ip"

    x = collecter_series([table], variable)

    ligne, lois = analyser_serie(
        x,
        "repetition",
        identifiant,
        variable,
        SEED + len(resultats_taille)
    )

    ligne.update(
        application=table.application.iat[0],
        reseau=table.reseau.iat[0],
        repetition=int(table.repetition.iat[0]),
        n_captures=1
    )

    resultats_taille.append(ligne)

    tracer_ajustement(
        ax,
        x,
        lois,
        "Taille des datagrammes IP",
        "Octets"
    )

    fig.suptitle(identifiant)

    enregistrer_figure(
        fig,
        f"taille_{identifiant}.png"
    )


# ------------------------------------------------------------
# 2. Analyse globale de chaque configuration
# ------------------------------------------------------------

for (application, reseau), tables in sorted(
    par_configuration.items()
):

    nom = f"{application}_{reseau}"

    fig, ax = plt.subplots(figsize=(9, 4.5))

    variable = "taille_ip"

    x = collecter_series(tables, variable)

    ligne, lois = analyser_serie(
        x,
        "configuration",
        nom,
        variable,
        SEED + len(resultats_taille)
    )

    ligne.update(
        application=application,
        reseau=reseau,
        repetition=np.nan,
        n_captures=len(tables)
    )

    resultats_taille.append(ligne)

    tracer_ajustement(
        ax,
        x,
        lois,
        "Taille des datagrammes IP",
        "Octets"
    )

    fig.suptitle(
        f"{nom} — {len(tables)} répétition(s)"
    )

    enregistrer_figure(
        fig,
        f"taille_{nom}_ensemble.png",
        afficher=True
    )


# ------------------------------------------------------------
# 3. Affichage et sauvegarde des résultats statistiques
# ------------------------------------------------------------

statistiques_tailles = pd.DataFrame(resultats_taille)

if not statistiques_tailles.empty:

    statistiques_tailles.to_csv(
        DOSSIER_TABLEAUX / "statistiques_tailles.csv",
        index=False
    )

    cols = [
        "niveau",
        "identifiant",
        "n_captures",
        "n",
        "m1",
        "m2",
        "variance",
        "ecart_type",
        "p_exponentielle",
        "p_normale",
        "p_uniforme",
        "meilleur_KS"
    ]

    display(statistiques_tailles[cols].round(5))

else:

    print("Aucun résultat : fichiers PCAP ou flux sélectionnés indisponibles.")

## 5. Gigue selon la définition IETF (RFC 3550)

La gigue RTP mesure la variation du temps de transit des paquets d'un même flux. Elle est calculée dans l'ordre de capture selon la RFC 3550 :

$D_i=(R_i-R_{i-1})-(S_i-S_{i-1})$ et $J_i=J_{i-1}+(|D_i|-J_{i-1})/16$.

Les temps de capture sont convertis en unités de l'horloge RTP, puis la gigue est exprimée en millisecondes. Les téléchargements LAPACK et les vidéos YouTube étudiés utilisent TCP/QUIC : cette définition ne s'applique pas directement à leurs paquets.

Pour les six captures Google Meet, les en-têtes RTP/SRTP sont extraits directement des datagrammes UDP/3478 (PCAP Ethernet/IPv4). Les flux sont séparés par adresses, ports, SSRC et type de payload. Les fréquences candidates (48 kHz pour les types 63 et 111, 90 kHz pour 96 et 98) sont propres à ces captures et validées empiriquement par la pente des horodatages. Elles ne sont pas confirmées par SDP. Les flux secondaires 97/99 et les flux insuffisamment cohérents sont exclus ; les seuils sont explicités dans le code. Le premier paquet initialise la gigue et n'est pas inclus dans les statistiques.

Les flux entrants servent à comparer la gigue observée à la réception ; les flux sortants décrivent les variations au point de capture. Les interarrivées restent une mesure distincte, utilisée dans la section 6.

In [ ]:
# 5.1 — Extraction RTP et calcul selon la RFC 3550
from collections import defaultdict
import struct
import socket

# Associations empiriques propres aux captures fournies, pas universelles.
HORLOGES_RTP_CANDIDATES = {63: 48000, 111: 48000, 96: 90000, 98: 90000}

def lire_rtp(fichier):
    flux = defaultdict(list)
    with open(fichier, 'rb') as f:
        entete = f.read(24)
        if entete[:4] != b'\xd4\xc3\xb2\xa1' or struct.unpack_from('<I', entete, 20)[0] != 1:
            raise ValueError('Format attendu : PCAP Ethernet little-endian, microsecondes')
        while h := f.read(16):
            sec, usec, taille, _ = struct.unpack('<IIII', h)
            p = f.read(taille)
            if len(p) < 42 or p[12:14] != b'\x08\x00' or p[23] != 17:
                continue
            if struct.unpack_from('!H', p, 20)[0] & 0x3fff:
                continue  # fragmentation IPv4
            o = 14 + (p[14] & 15)*4
            if len(p) < o+8:
                continue
            sp, dp, longueur, _ = struct.unpack_from('!HHHH', p, o)
            if 3478 not in (sp, dp) or len(p) < o+longueur:
                continue
            d = p[o+8:o+longueur]
            if len(d) < 12 or d[0] >> 6 != 2 or 192 <= d[1] <= 223:
                continue  # exclut STUN/DTLS et RTCP/SRTCP multiplexés
            fin = 12 + 4*(d[0] & 15)
            if d[0] & 16:
                if len(d) < fin+4:
                    continue
                fin += 4 + 4*struct.unpack_from('!H', d, fin+2)[0]
            if len(d) < fin:
                continue
            pt = d[1] & 127
            seq, ts, ssrc = struct.unpack_from('!HII', d, 2)
            src = socket.inet_ntoa(p[26:30]); dst = socket.inet_ntoa(p[30:34])
            cle = (src, dst, sp, dp, f'0x{ssrc:08x}', pt)
            flux[cle].append((sec*1000000 + usec, seq, ts))
    return flux

def calcul_gigue_ietf(temps_s, timestamps, frequence):
    # L'ordre est celui des paquets dans la capture, pas celui des séquences.
    delta_R = np.diff(temps_s) * frequence
    delta_S = (np.diff(timestamps.astype(np.int64)) + 2**31) % 2**32 - 2**31
    D = delta_R - delta_S
    J = np.zeros(len(temps_s))
    for i, d in enumerate(D, 1):
        J[i] = J[i-1] + (abs(d) - J[i-1])/16
    J[0] = np.nan  # initialisation, pas une mesure
    return 1000*J/frequence

def analyser_gigue_pcap(dossier):
    inventaire, stats, series = [], [], []
    for fichier in sorted(Path(dossier).glob('googlemeet_*.pcap')):
        for (src, dst, sp, dp, ssrc, pt), paquets in lire_rtp(fichier).items():
            if len(paquets) < 2:
                continue
            a = np.asarray(paquets, dtype=np.int64)
            t = (a[:,0]-a[0,0])/1e6
            timestamps = a[:,2]
            delta_S = (np.diff(timestamps)+2**31)%2**32-2**31
            deplies = np.r_[0, np.cumsum(delta_S)]
            pente = np.polyfit(t, deplies, 1)[0] if np.ptp(t)>0 else np.nan
            seq_ok = np.mean((np.diff(a[:,1])%65536)==1)
            direction = 'entrant' if sp == 3478 else 'sortant'
            frequence = HORLOGES_RTP_CANDIDATES.get(pt)
            info = dict(capture_id=fichier.stem, reseau='Ethernet' if 'ethernet' in fichier.stem else 'Wi-Fi',
                        direction=direction, src_ip=src, dst_ip=dst, src_port=sp, dst_port=dp,
                        rtp_ssrc=ssrc, payload_type=pt, nb_paquets=len(a), duree_s=t[-1],
                        horloge_estimee_Hz=pente, sequences_consecutives_pct=100*seq_ok,
                        horloge_retenue_Hz=frequence)
            # Critères empiriques conservateurs : au moins 100 paquets, 10 s,
            # >95 % de séquences consécutives, pente à moins de 1 % de la fréquence.
            # Ne pas appliquer aux flux secondaires non identifiés (97/99).
            valide = (frequence is not None and len(a)>=100 and t[-1]>=10 and
                      seq_ok>=.95 and abs(pente/frequence-1)<=.01 and np.all(np.diff(t)>=0))
            inventaire.append(dict(info, statut='retenu' if valide else 'exclu'))
            if not valide:
                continue
            j = calcul_gigue_ietf(t, timestamps, frequence)
            g = pd.Series(j[1:])
            media = 'Audio' if frequence==48000 else 'Video'
            stats.append(dict(info, media_probable=media, nb_mesures=len(g), moyenne_ms=g.mean(),
                              mediane_ms=g.median(), ecart_type_ms=g.std(), min_ms=g.min(),
                              max_ms=g.max(), p95_ms=g.quantile(.95), p99_ms=g.quantile(.99), finale_ms=g.iloc[-1]))
            series.append(pd.DataFrame(dict(capture_id=fichier.stem, direction=direction, media_probable=media,
                                            rtp_ssrc=ssrc, payload_type=pt, src_ip=src, dst_ip=dst, src_port=sp, dst_port=dp,
                                            reseau=info['reseau'], temps_s=t, rtp_seq=a[:,1],
                                            rtp_timestamp=timestamps, gigue_ietf_ms=j)))
    return pd.DataFrame(inventaire), pd.DataFrame(stats), pd.concat(series, ignore_index=True) if series else pd.DataFrame()



In [ ]:
# 5.2 — Calcul sur les captures Google Meet et export des résultats
df_controle_gigue, df_stats_gigue, df_series_gigue = analyser_gigue_pcap(DOSSIER_PCAP)

if not df_controle_gigue.empty:
    display(df_controle_gigue.round(3))
    df_controle_gigue.to_csv(DOSSIER_TABLEAUX / "inventaire_flux_rtp.csv", index=False)
else:
    print("Aucun flux RTP candidat dans les PCAP Google Meet.")

if not df_stats_gigue.empty:
    display(df_stats_gigue.round(3))
    df_stats_gigue.to_csv(DOSSIER_TABLEAUX / "statistiques_gigue_ietf.csv", index=False)
    df_series_gigue.to_csv(DOSSIER_TABLEAUX / "series_gigue_ietf.csv", index=False)
else:
    print("Aucun flux ne satisfait les critères de validation.")

# Contrôle par capture, y compris les applications hors RTP
controle_par_capture = []
for info in inventaire:
    identifiant = info["capture_id"]
    if info["application"] != "googlemeet":
        statut = "Non applicable (TCP/QUIC)"
        n_flux = 0
    else:
        n_flux = int(df_stats_gigue["capture_id"].eq(identifiant).sum()) if not df_stats_gigue.empty else 0
        statut = "Calcul effectué (horloge empirique)" if n_flux else "Aucun flux retenu"
    controle_par_capture.append({
        "capture_id": identifiant, "nb_flux_retenus": n_flux, "statut": statut
    })
df_controle_captures_gigue = pd.DataFrame(controle_par_capture)
display(df_controle_captures_gigue)
df_controle_captures_gigue.to_csv(DOSSIER_TABLEAUX / "controle_captures_gigue.csv", index=False)


In [ ]:
# 5.3 — Comparaison par répétition, puis par réseau
# Les moyennes de flux sont agrégées par capture avant de comparer les réseaux.
# Chaque répétition a le même poids, quel que soit son nombre de paquets.
if not df_stats_gigue.empty:
    gigue_par_capture = (
        df_stats_gigue.groupby(
            ["capture_id", "reseau", "direction", "media_probable"], as_index=False
        ).agg(
            nb_flux=("moyenne_ms", "size"),
            nb_paquets=("nb_paquets", "sum"),
            gigue_moyenne_ms=("moyenne_ms", "mean")
        )
    )
    comparaison_gigue = (
        gigue_par_capture.groupby(
            ["reseau", "direction", "media_probable"], as_index=False
        ).agg(
            nb_captures=("capture_id", "nunique"),
            moyenne_ms=("gigue_moyenne_ms", "mean"),
            ecart_type_repetitions_ms=("gigue_moyenne_ms", "std")
        )
    )
    display(gigue_par_capture.round(3))
    display(comparaison_gigue.round(3))
    gigue_par_capture.to_csv(DOSSIER_TABLEAUX / "gigue_par_capture.csv", index=False)
    comparaison_gigue.to_csv(DOSSIER_TABLEAUX / "comparaison_gigue_reseaux.csv", index=False)
else:
    gigue_par_capture = pd.DataFrame()
    comparaison_gigue = pd.DataFrame()


In [ ]:
# 5.4 — Histogrammes et évolution temporelle (flux entrants)
if not df_series_gigue.empty:
    entrants = df_series_gigue[df_series_gigue["direction"].eq("entrant")]
    for media, table in entrants.groupby("media_probable"):
        valeurs = table["gigue_ietf_ms"].dropna().to_numpy()
        if not len(valeurs):
            continue
        # Mêmes bornes pour les deux réseaux, toutes les valeurs incluses.
        bins = np.linspace(0, max(float(valeurs.max()), 1e-9), 61)
        fig, ax = plt.subplots(figsize=(9, 4.5))
        for reseau, groupe in table.groupby("reseau"):
            x = groupe["gigue_ietf_ms"].dropna()
            ax.hist(x, bins=bins, density=True, histtype="step", linewidth=1.5, label=reseau)
        ax.set(xlabel="Gigue IETF (ms)", ylabel="Densité de probabilité",
               title=f"Distribution de la gigue entrante — {media}")
        ax.legend()
        enregistrer_figure(fig, f"gigue_ietf_hist_{media.lower()}.png", afficher=True)

    for (capture, media), table in entrants.groupby(["capture_id", "media_probable"]):
        fig, ax = plt.subplots(figsize=(10, 4.5))
        for cle, flux in table.groupby(
            ["src_ip", "dst_ip", "src_port", "dst_port", "rtp_ssrc", "payload_type"]
        ):
            ax.plot(flux["temps_s"], flux["gigue_ietf_ms"], linewidth=0.8,
                    label=f"SSRC {cle[4]} / PT {cle[5]}")
        ax.set(xlabel="Temps depuis le premier paquet du flux (s)", ylabel="Gigue IETF (ms)",
               title=f"{capture} — {media} entrant")
        ax.legend()
        enregistrer_figure(fig, f"gigue_ietf_{capture}_{media.lower()}.png")

if not gigue_par_capture.empty:
    video = gigue_par_capture[
        gigue_par_capture["direction"].eq("entrant") &
        gigue_par_capture["media_probable"].eq("Video")
    ]
    if not video.empty:
        fig, ax = plt.subplots(figsize=(10, 4.5))
        ax.bar(video["capture_id"], video["gigue_moyenne_ms"])
        ax.set(ylabel="Gigue moyenne (ms)", title="Gigue vidéo entrante par expérience")
        ax.tick_params(axis="x", labelrotation=35)
        enregistrer_figure(fig, "comparaison_gigue_video_entrante.png", afficher=True)


## 6. Arrivée des paquets d'information — interarrivées, moments et ajustements

In [ ]:
resultats_arrivees = []
for identifiant, table in captures.items():
    x = collecter_series([table], "interarrivee_s", conversion=1000)
    ligne, lois = analyser_serie(x, "repetition", identifiant, "interarrivee_ms",
                                SEED + len(resultats_arrivees))
    ligne.update(application=table.application.iat[0], reseau=table.reseau.iat[0],
                 repetition=int(table.repetition.iat[0]), n_captures=1)
    resultats_arrivees.append(ligne)
    fig, ax = plt.subplots(figsize=(9, 4.5))
    tracer_ajustement(ax, x, lois, "Temps interarrivées positifs", "Millisecondes")
    fig.suptitle(identifiant)
    enregistrer_figure(fig, f"interarrivees_{identifiant}.png")

for (application, reseau), tables in sorted(par_configuration.items()):
    nom = f"{application}_{reseau}"
    x = collecter_series(tables, "interarrivee_s", conversion=1000)
    ligne, lois = analyser_serie(x, "configuration", nom, "interarrivee_ms",
                                SEED + len(resultats_arrivees))
    ligne.update(application=application, reseau=reseau, repetition=np.nan,
                 n_captures=len(tables))
    resultats_arrivees.append(ligne)
    fig, ax = plt.subplots(figsize=(9, 4.5))
    tracer_ajustement(ax, x, lois, "Temps interarrivées positifs", "Millisecondes")
    fig.suptitle(f"{nom} — {len(tables)} répétition(s)")
    enregistrer_figure(fig, f"interarrivees_{nom}_ensemble.png", afficher=True)

statistiques_arrivees = pd.DataFrame(resultats_arrivees)
if not statistiques_arrivees.empty:
    statistiques_arrivees.to_csv(DOSSIER_TABLEAUX / "statistiques_interarrivees.csv", index=False)
    display(statistiques_arrivees[["niveau", "identifiant", "n_captures", "n", "m1", "m2",
                                   "fraction_zero", "p_exponentielle", "p_normale", "p_uniforme",
                                   "meilleur_KS"]].round(5))
else:
    print("Aucun processus d'arrivée analysable.")

## 7. Apprentissage machine — nettoyage, séparation et classification

In [ ]:
# Les classes « application » sont les activités de capture, pas les protocoles.
# La classification données/contrôle ne dispose d'une vérité terrain fiable que pour TCP
# et les contrôles identifiables ; l'UDP/QUIC chiffré demeure « indéterminé ».


def construire_ml(toutes_captures):
    liste = []
    generateur = np.random.default_rng(SEED)
    for identifiant, paquet in toutes_captures.items():
        if paquet.empty:
            continue
        df = paquet.copy()
        if MAX_PAQUETS_ML is not None and len(df) > MAX_PAQUETS_ML:
            indices = np.sort(generateur.choice(len(df), MAX_PAQUETS_ML, replace=False))
            df = df.iloc[indices].copy()
        df["interarrivee_ms"] = df["interarrivee_s"].clip(lower=0) * 1000
        df["cible_type"] = df["type_paquet"].map({
            "tcp_donnees": "donnees", "tcp_controle": "controle",
            "controle_identifiable": "controle"
        })
        colonnes = ["capture_id", "application", "reseau", "repetition", "transport",
                    "taille_ip", "charge_utile_octets", "interarrivee_ms",
                    "paquets_par_s", "tcp_syn", "tcp_fin", "tcp_rst", "tcp_ack",
                    "cible_type"]
        liste.append(df[colonnes])
    if not liste:
        return pd.DataFrame()
    resultat = pd.concat(liste, ignore_index=True)
    numeros = ["taille_ip", "charge_utile_octets", "interarrivee_ms", "paquets_par_s",
               "tcp_syn", "tcp_fin", "tcp_rst", "tcp_ack"]
    resultat[numeros] = resultat[numeros].replace([np.inf, -np.inf], np.nan)
    resultat.loc[resultat["taille_ip"] <= 0, "taille_ip"] = np.nan
    resultat.loc[resultat["interarrivee_ms"] < 0, "interarrivee_ms"] = np.nan
    return resultat


def repartir_captures(df):

    # Identifier les captures indépendantes
    groupes = df[
        ["capture_id", "application", "reseau", "repetition"]
    ].drop_duplicates()

    affectation = {}

    # Répartition pour chaque application et chaque réseau
    for (application, reseau), bloc in groupes.groupby(
        ["application", "reseau"]
    ):

        ids = bloc.sort_values(
            ["repetition", "capture_id"]
        ).capture_id.tolist()

        # Première capture : entraînement
        # Deuxième capture : validation
        # Troisième capture : test
        for identifiant, jeu in zip(
            ids, ("train", "validation", "test")
        ):
            affectation[identifiant] = jeu

        # Captures supplémentaires : entraînement
        for identifiant in ids[3:]:
            affectation[identifiant] = "train"

    

    resultat = df.copy()

    resultat["jeu"] = resultat["capture_id"].map(affectation)

    resultat = resultat.dropna(subset=["jeu"]).copy()

    # Vérification : aucune capture partagée entre les jeux
    for a in ("train", "validation", "test"):
        for b in ("train", "validation", "test"):
            if a != b:
                assert set(
                    resultat.loc[resultat.jeu.eq(a), "capture_id"]
                ).isdisjoint(
                    resultat.loc[resultat.jeu.eq(b), "capture_id"]
                )

    return resultat


def construire_pipeline(modele, colonnes_numeriques, colonnes_categorielles):
    transformations = [("numerique", Pipeline([
        ("imputation", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("standardisation", StandardScaler())]), colonnes_numeriques)]
    if colonnes_categorielles:
        transformations.append(("categoriel", Pipeline([
            ("imputation", SimpleImputer(strategy="most_frequent")),
            ("encodage", OneHotEncoder(handle_unknown="ignore"))]), colonnes_categorielles))
    return Pipeline([("pretraitement", ColumnTransformer(transformations)), ("modele", modele)])

ml = construire_ml(captures)
if not ml.empty:
    ml = repartir_captures(ml)
    display(pd.crosstab([ml.application, ml.reseau], ml.jeu))
    print("Paquets IP analysés :", f"{len(ml):,}")
    print("Type données/contrôle étiquetable :", f"{ml.cible_type.notna().mean():.1%}")
else:
    print("ML indisponible sans captures PCAP.")

In [ ]:
def classer_paquets(df, cible, variables_numeriques,
                   variables_categorielles, etiquette):

    # ------------------------------------------------------------
    # 1. Préparation des données
    # ------------------------------------------------------------

    donnees = df.dropna(subset=[cible, "jeu"]).copy()

    if donnees.empty:
        print(f"{etiquette} : aucune donnée étiquetée.")
        return pd.DataFrame(), {}

    masques = {
        jeu: donnees.jeu.eq(jeu)
        for jeu in ("train", "validation", "test")
    }

    X = donnees[variables_numeriques + variables_categorielles]
    y = donnees[cible]

    if (
        y[masques["train"]].nunique() < 2
        or any(not masques[j].any() for j in ("validation", "test"))
    ):
        print(f"{etiquette} : classes ou jeux insuffisants pour une évaluation.")
        return pd.DataFrame(), {}

    # ------------------------------------------------------------
    # 2. Définition des modèles
    # ------------------------------------------------------------

    modeles = {
        "majoritaire": DummyClassifier(
            strategy="most_frequent"
        ),

        "regression_logistique": LogisticRegression(
            max_iter=1500,
            class_weight="balanced",
            random_state=SEED
        ),

        "regression_lineaire_ridge": RidgeClassifier(
            class_weight="balanced"
        )
    }

    performances = []
    entraines = {}
    classes = sorted(y.unique())

    # ------------------------------------------------------------
    # 3. Entraînement et évaluation des modèles
    # ------------------------------------------------------------

    for nom, estimateur in modeles.items():

        pipeline = construire_pipeline(
            estimateur,
            variables_numeriques,
            variables_categorielles
        )

        # Entraînement
        pipeline.fit(
            X.loc[masques["train"]],
            y.loc[masques["train"]]
        )

        entraines[nom] = pipeline

        # Validation et test
        for jeu in ("validation", "test"):

            masque = masques[jeu]

            predictions = pipeline.predict(X.loc[masque])

            scores = {
                "cible": etiquette,
                "modele": nom,
                "jeu": jeu,

                "accuracy": accuracy_score(
                    y[masque], predictions
                ),

                "f1_macro": f1_score(
                    y[masque],
                    predictions,
                    labels=classes,
                    average="macro",
                    zero_division=0
                ),

                "n_paquets": int(masque.sum()),

                "n_captures": int(
                    donnees.loc[masque, "capture_id"].nunique()
                ),

                "classes_reelles": ", ".join(
                    sorted(y[masque].unique())
                )
            }

            performances.append(scores)

    # ------------------------------------------------------------
    # 4. Affichage des performances
    # ------------------------------------------------------------

    tableau = pd.DataFrame(performances)

    tableau.to_csv(
        DOSSIER_TABLEAUX / f"metriques_ml_{etiquette}.csv",
        index=False
    )

    display(tableau.round(4))

    # ------------------------------------------------------------
    # 5. Modèle utilisé pour les matrices de confusion
    # ------------------------------------------------------------

    nom_modele = "regression_logistique"

    for jeu in ("validation", "test"):

        masque = masques[jeu]

        prediction = entraines[nom_modele].predict(
            X.loc[masque]
        )

        print(f"{etiquette} — {jeu} — {nom_modele} :")

        print(classification_report(
            y.loc[masque],
            prediction,
            labels=classes,
            zero_division=0
        ))

        fig, ax = plt.subplots(figsize=(6.4, 5.0))

        ConfusionMatrixDisplay.from_predictions(
            y.loc[masque],
            prediction,
            labels=classes,
            ax=ax,
            xticks_rotation=25,
            cmap="Blues",
            values_format="d"
        )

        # Titre contenant le nom du modèle
        ax.set_title(
            f"{etiquette} — {jeu}\n"
            f"{nom_modele.replace('_', ' ').capitalize()}"
        )

        ax.set_xlabel("Classe prédite")
        ax.set_ylabel("Classe réelle")

        enregistrer_figure(
            fig,
            f"confusion_{etiquette}_{jeu}_{nom_modele}.png",
            afficher=True
        )

    # ------------------------------------------------------------
    # 6. Comparaison des performances des trois modèles
    # ------------------------------------------------------------

    fig, ax = plt.subplots(figsize=(9, 4))

    rang = tableau.pivot(
        index="modele",
        columns="jeu",
        values="f1_macro"
    )

    rang.plot.bar(
        ax=ax,
        ylim=(0, 1),
        rot=10
    )

    ax.set(
        xlabel="Modèle",
        ylabel="F1 macro",
        title=f"Comparaison ML — {etiquette}"
    )

    enregistrer_figure(
        fig,
        f"comparaison_ml_{etiquette}.png",
        afficher=True
    )

    return tableau, entraines


# ------------------------------------------------------------
# 7. Exécution des classifications
# ------------------------------------------------------------

if not ml.empty:

    # Classification des applications
    performances_application, modeles_application = classer_paquets(
        ml,
        "application",
        [
            "taille_ip",
            "charge_utile_octets",
            "interarrivee_ms",
            "paquets_par_s",
            "tcp_syn",
            "tcp_fin",
            "tcp_rst",
            "tcp_ack"
        ],
        ["transport"],
        "application"
    )

    # Classification des paquets données/contrôle
    performances_type, modeles_type = classer_paquets(
        ml,
        "cible_type",
        [
            "taille_ip",
            "interarrivee_ms",
            "paquets_par_s",
            "tcp_syn",
            "tcp_fin",
            "tcp_rst",
            "tcp_ack"
        ],
        ["transport"],
        "type_paquet"
    )

else:

    print("Classification indisponible : aucune donnée.")

### Limites de la classification

Pour la première tâche, la classe est **l'activité ayant produit la capture**, et non une identification certaine de chaque paquet de cette application (trafic d'arrière-plan possible). Pour la seconde, seuls les paquets TCP et les contrôles identifiables servent de vérité terrain : l'UDP/QUIC chiffré reste non étiqueté. Avec **une seule capture LAPACK par réseau**, la validation à trois classes ne comporte pas LAPACK et le test indépendant de cette classe ne repose que sur une capture ; il faut de nouvelles captures LAPACK pour une évaluation robuste.

# 8 reconnaisance de traffic 


In [ ]:
# ------------------------------------------------------------
# 8. Identification automatique des scénarios
# ------------------------------------------------------------

DOSSIER_SCENARIOS = RACINE / "data" / "scenarios"
DOSSIER_SCENARIOS.mkdir(parents=True, exist_ok=True)

SEUIL_CONFIANCE = 0.8
MIN_PAQUETS_FLUX = 1000

# Mêmes variables que celles utilisées pendant l'entraînement
VARIABLES_NUMERIQUES = [
    "taille_ip",
    "charge_utile_octets",
    "interarrivee_ms",
    "paquets_par_s",
    "tcp_syn",
    "tcp_fin",
    "tcp_rst",
    "tcp_ack"
]

VARIABLES_CATEGORIELLES = ["transport"]

VARIABLES_ML = VARIABLES_NUMERIQUES + VARIABLES_CATEGORIELLES


def preparer_flux_ml(df):

    df = df.copy()

    # Même calcul que dans construire_ml()
    df["interarrivee_ms"] = (
        df["interarrivee_s"].clip(lower=0) * 1000
    )

    # Nettoyage des variables numériques
    df[VARIABLES_NUMERIQUES] = df[
        VARIABLES_NUMERIQUES
    ].replace([np.inf, -np.inf], np.nan)

    df.loc[df["taille_ip"] <= 0, "taille_ip"] = np.nan

    # Conserver les conversations TCP et UDP identifiables
    df = df[df["transport"].isin(["TCP", "UDP"])]

    df = df.dropna(subset=["stream_id", "temps_s"])

    return df

In [ ]:
# ------------------------------------------------------------
# Réutilisation du modèle déjà entraîné dans la section 7
# ------------------------------------------------------------

modele_scenario = modeles_application["regression_logistique"]

print("Modèle utilisé : régression logistique")
print("Classes reconnues :", modele_scenario.classes_)
print("Variables utilisées :", modele_scenario.feature_names_in_)

In [ ]:
def identifier_conversations(df, modele):

    df = preparer_flux_ml(df)

    if df.empty:
        return pd.DataFrame()

    # --------------------------------------------------------
    # 1. Prédiction de chaque paquet
    # --------------------------------------------------------

    df["prediction"] = modele.predict(
        df[VARIABLES_ML]
    )

    probabilites = modele.predict_proba(
        df[VARIABLES_ML]
    )

    classes = modele.classes_

    for i, classe in enumerate(classes):

        df[f"proba_{classe}"] = probabilites[:, i]

    # --------------------------------------------------------
    # 2. Regroupement par conversation
    # --------------------------------------------------------

    resultats = []

    for (transport, stream_id), flux in df.groupby(
        ["transport", "stream_id"]
    ):

        # Probabilité moyenne de chaque application
        scores = {
            classe: flux[f"proba_{classe}"].mean()
            for classe in classes
        }

        # Application ayant le meilleur score
        application_predite = max(scores, key=scores.get)

        confiance = scores[application_predite]

        nombre_paquets = len(flux)

        # Rejeter les conversations trop courtes ou incertaines
        if (
            confiance < SEUIL_CONFIANCE
            or nombre_paquets < MIN_PAQUETS_FLUX
        ):

            application_predite = "inconnu"

        # Durée de la conversation
        duree = flux["temps_s"].max() - flux["temps_s"].min()

        # Volume total de données IP
        volume_octets = flux["taille_ip"].sum()

        resultats.append({

            "transport": transport,
            "stream_id": int(stream_id),

            "ip_a": flux["ip_src"].iloc[0],
            "ip_b": flux["ip_dst"].iloc[0],

            "port_a": flux["port_src"].iloc[0],
            "port_b": flux["port_dst"].iloc[0],

            "application_predite": application_predite,
            "confiance_pct": confiance * 100,

            "nombre_paquets": nombre_paquets,
            "volume_Mo": volume_octets / 1e6,
            "duree_s": duree,

            "score_youtube": scores.get("youtube", 0),
            "score_googlemeet": scores.get("googlemeet", 0),
            "score_lapack": scores.get("lapack", 0)

        })

    tableau = pd.DataFrame(resultats)

    return tableau.sort_values(
        "volume_Mo",
        ascending=False
    ).reset_index(drop=True)

In [ ]:
def identifier_scenario(conversations):

    if conversations.empty:
        return "Scénario indéterminé"

    applications = set(
        conversations["application_predite"]
    )

    applications.discard("inconnu")

    if applications == {"youtube"}:

        return "Scénario 1 : YouTube"

    elif applications == {"lapack"}:

        return "Scénario 2 : Téléchargement"

    elif applications == {"googlemeet"}:

        return "Scénario 3 : Google Meet"

    elif applications == {"youtube", "lapack", "googlemeet"}:

        return "Scénario 4 : Les trois activités simultanées"

    elif len(applications) > 1:

        return "Trafic mixte partiel : " + ", ".join(
            sorted(applications)
        )

    else:

        return "Scénario indéterminé"

In [ ]:
# ------------------------------------------------------------
# Analyse automatique des nouvelles captures
# ------------------------------------------------------------

fichiers_scenarios = sorted([
    fichier
    for fichier in DOSSIER_SCENARIOS.iterdir()
    if fichier.suffix.lower() in {".pcap", ".pcapng"}
])

if TSHARK is None:
    raise RuntimeError("TShark introuvable.")

champs_disponibles = obtenir_champs_tshark(TSHARK)

resultats_scenarios = []
conversations_scenarios = {}

for fichier in fichiers_scenarios:

    print("\n" + "=" * 70)
    print("CAPTURE :", fichier.name)
    print("=" * 70)

    # Extraction de tous les paquets IP
    df = extraire_pcap(
        fichier,
        TSHARK,
        champs_disponibles
    )

    df = convertir_numerique(df)
    df = etiqueter_paquets(df)

    # Identification automatique des conversations
    conversations = identifier_conversations(
        df,
        modele_scenario
    )

    # Identification du scénario
    scenario_predit = identifier_scenario(conversations)

    conversations_scenarios[fichier.stem] = conversations

    print("\nSCÉNARIO PRÉDIT :", scenario_predit)
    print("Nombre total de paquets IP :", len(df))
    print("Nombre de conversations :", len(conversations))

    if not conversations.empty:

        colonnes = [
            "transport",
            "stream_id",
            "ip_a",
            "ip_b",
            "application_predite",
            "confiance_pct",
            "nombre_paquets",
            "volume_Mo"
        ]

        display(conversations[colonnes].round(3))

        conversations.to_csv(
            DOSSIER_TABLEAUX /
            f"classification_{fichier.stem}.csv",
            index=False
        )

        nombre_classifiables = int(
            conversations["nombre_paquets"].sum()
        )

        nombre_inconnus = int(
            conversations.loc[
                conversations["application_predite"] == "inconnu",
                "nombre_paquets"
            ].sum()
        )

    else:

        nombre_classifiables = 0
        nombre_inconnus = 0

    resultats_scenarios.append({

        "capture": fichier.name,
        "scenario_predit": scenario_predit,
        "n_paquets_ip": len(df),
        "n_paquets_analyses": nombre_classifiables,
        "n_paquets_hors_flux": len(df) - nombre_classifiables,
        "n_paquets_inconnus": nombre_inconnus,
        "n_conversations": len(conversations)

    })


# Tableau récapitulatif
tableau_scenarios = pd.DataFrame(resultats_scenarios)

display(tableau_scenarios)

tableau_scenarios.to_csv(
    DOSSIER_TABLEAUX / "resultats_scenarios.csv",
    index=False
)

In [ ]:
for identifiant, conversations in conversations_scenarios.items():

    if conversations.empty:
        continue

    repartition = conversations.groupby(
        "application_predite"
    )["nombre_paquets"].sum()

    repartition = repartition.reindex(
        ["youtube", "googlemeet", "lapack", "inconnu"],
        fill_value=0
    )

    fig, ax = plt.subplots(figsize=(9, 4.5))

    repartition.plot.bar(
        ax=ax,
        color=["red", "seagreen", "steelblue", "gray"],
        rot=0
    )

    ax.set(
        title=f"{identifiant}\nClassification par régression logistique",
        xlabel="Application prédite",
        ylabel="Nombre de paquets"
    )

    ax.bar_label(ax.containers[0])

    enregistrer_figure(
        fig,
        f"prediction_{identifiant}.png",
        afficher=True
    )